1. Web Scraping [20 puntos]:
- U@lizar técnicas de Web Scraping para extraer información de al menos 500 bebidas alcohólicas del
si@o web hGps://dis@ller.com/search.
- Los datos a obtener para cada bebida están definidos por las siguientes columnas que se pueden
extraer de la página asociada a la bebida:
columns=['Name', 'Type', 'Cask', 'Location', 'Age', 'ABV %', 'Cost',
'Badge', '# Ratings', "Community Rating", 'Flavor Summary', 'Expert',
'Expert Score', 'Smoky', 'Earthy', 'Spicy', 'Herbal', 'Oily', 'Bitter',
'Rich', 'Sweet', 'Mineral', 'Salty', 'Umami', 'Tart', 'Fruity',
'Floral', 'Review'])
- Almacenar los datos obtenidos en un DataFrame de Pandas. Los datos deberían verse como en la
Figura 2 (aproximadamente).
Figura 2. Ejemplo de la planilla de datos obtenida con Web Scraping.

In [ ]:
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC 
from selenium.webdriver.common.by import By
from selenium.webdriver import Chrome
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import json

import pandas as pd

In [ ]:
df = pd.DataFrame(columns=['name', 'type', 'cask', 'location', 'age', 'abv %', 'cost', 
'badge', '# ratings', "community rating", 'flavor summary', 'expert', 
'expert score', 'smoky', 'earthy', 'spicy', 'herbal', 'oily', 'bitter', 
'rich', 'sweet', 'mineral', 'salty', 'umami', 'tart', 'fruity', 
'floral'])

In [ ]:
def startDriver() -> webdriver.Chrome:
	service = Service(ChromeDriverManager().install())
	options = Options()
	options.add_argument("--disable-extensions")
	options.add_argument("--headless")
	driver = Chrome(service=service, options=options)
	return driver

driver = startDriver()

In [ ]:
url = "https://distiller.com/search?page="
driver.get(url)
WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button.align-right.secondary.slidedown-button"))).click()
WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button.banner__close.privacy__banner"))).click()

WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button.spirit-family-select__button.js-select-spirit-family.all.selected"))).click()
WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[data-value="agave"]'))).click()

hrefs = []

for x in range(1, 51):
    driver.get(f"{url}{x}")
    elements = driver.find_elements(By.CSS_SELECTOR, "li.spirit.agave-content")
    for element in elements:
        hrefs.append(element.find_element(By.TAG_NAME, "a").get_attribute("href"))

for href in hrefs:
    print(href)

driver.quit()

In [ ]:
driver = startDriver()
for href in hrefs:
    driver.get(href)
    
    #Nombre
    try:
        name = driver.find_element(By.XPATH, "/html/body/div[7]/div/div/main/div/div[1]/div[1]/div[3]/h1").text
        if name == "":
            name = "unknown"
    except:
        name = "unknown"
    #Tipo

    try:   
        types = driver.find_element(By.XPATH,"/html/body/div[7]/div/div/main/div/div[1]/div[1]/div[3]/div/p[1]").text
        if types == "":
            types = pd.NA
    except:
        types = pd.NA
    #Cask
    try:
        cask = driver.find_element(By.XPATH,"/html/body/div[7]/div/div/main/div/div[1]/div[2]/div[1]/div/div/div[2]/div[3]/ul/li[3]/div[2]").text
        if cask == "":
            cask = "unknown"    
    except:
        cask = "unknown"
    #Location y badge

    try:
        temp = driver.find_element(By.XPATH,"/html/body/div[7]/div/div/main/div/div[1]/div[1]/div[3]/div/p[2]").text.split("// ")

        location = temp[1]
        badge = temp[0]

        if location == "":
            location = "unknown"
        if badge == "":
            badge = "unknown"
    except:
        location = "unknown"
        badge = "unknown"

    #age

    try:
        age = driver.find_element(By.XPATH,"/html/body/div[7]/div/div/main/div/div[1]/div[2]/div[1]/div/div/div[2]/div[3]/ul/li[1]/ul/li[1]/div[2]").text
        if age == "":
            age = "unknown"
    except:
        age = "unknown"

    #ABV
    try:
        abv = driver.find_element(By.XPATH,"/html/body/div[7]/div/div/main/div/div[1]/div[2]/div[1]/div/div/div[2]/div[3]/ul/li[1]/ul/li[3]/div[2]").text
        if abv == "":
            abv = pd.NA 
    except:
        abv = pd.NA

    #Cost
    try:
        cost = driver.find_element(By.XPATH, "/html/body/div[7]/div/div/main/div/div[1]/div[2]/div[1]/div/div/div[2]/div[3]/ul/li[1]/ul/li[2]/div[2]/div").get_attribute("class")[-1]
        if cost == "":  
            cost = pd.NA
    except:
        cost = pd.NA

    #Ratings
    try:
        ratings = driver.find_element(By.XPATH,"/html/body/div[7]/div/div/main/div/div[1]/div[2]/div[1]/div/div/div[1]/div/div/div/div/div[3]/a/span[2]").text
        if ratings == "":
            ratings = "unknown"
    except:
        ratings = "unknown"

    #Community Rating
    try:
        community_rating = driver.find_element(By.XPATH,"/html/body/div[7]/div/div/main/div/div[1]/div[2]/div[1]/div/div/div[1]/div/div/div/div/div[1]/span").text
        if community_rating == "" or community_rating == "No one has reviewed this yet. Be the first":
            community_rating = "unknown"
    except:
        community_rating = "unknown"

    #Flavor Summary
    try:
        flavor_summary = driver.find_element(By.XPATH,"/html/body/div[7]/div/div/main/div/div[1]/div[2]/div[1]/div/div/div[5]/h3").text
        if flavor_summary == "":
            flavor_summary = "unknown"
    except:
        flavor_summary = "unknown"  

    #Expert
    try:
        expert = driver.find_element(By.XPATH,"/html/body/div[7]/div/div/main/div/div[1]/div[2]/div[1]/div/div/div[3]/div[2]/div[1]/a").text
        if expert == "":
            expert = "unknown"
    except:
        expert = "unknown"

    #Expert Score
    try:
        expert_score = driver.find_element(By.XPATH,"/html/body/div[7]/div/div/main/div/div[1]/div[2]/div[1]/div/div/div[3]/div[2]/div[2]/span").text
        if expert_score == "":
            expert_score = "unknown"
    except:
        expert_score = "unknown"
    

    #Flavor Profile
    canvas_element = driver.find_element(By.XPATH, "/html/body/div[7]/div/div/main/div/div[1]/div[2]/div[1]/div/div/div[5]/canvas")
    data_flavors = canvas_element.get_attribute("data-flavors")
    
    df_2 = pd.DataFrame([json.loads(data_flavors)])

    #Create df_1
    df_1 = {
        'name': name,
        'type': types,
        'cask': cask,
        'location': location,
        'age': age,
        'abv %': abv,
        'cost': cost,
        'badge': badge,
        '# ratings': ratings,
        'community rating': community_rating,
        'flavor summary': flavor_summary,
        'expert': expert,
        'expert score': expert_score
    }

    df_1 = pd.DataFrame([df_1])
    

    #Concatenate df_1 and df_2
    df_1 = pd.concat([df_1, df_2], axis=1)

    #Append to df
    df = pd.concat([df, df_1], ignore_index=True)


driver.quit()
df = df.dropna()
df.to_csv("distiler.csv", index=False)
    


2. Preprocesamiento de datos [20 puntos]:
- Realizar una exploración inicial de los datos obtenidos mediante estadís@cas básicas y visualización
de cada variable relevante, iden@ficando posibles valores faltantes, outliers o inconsistencias.
- Aplicar técnicas de preprocesamiento según sea necesario.
- Dividir los datos en conjuntos de entrenamiento y prueba, con una proporción 80/20 o 70/30 según
sea necesario/convenient.

3. Regresión Lineal [20 puntos]:
- Entrenar un modelo de Regresión Lineal u@lizando las variables predictoras seleccionadas para
predecir el ra@ng de las bebidas.
- Evaluar el rendimiento del modelo u@lizando métricas como el Mean Squared Error (MSE) y el
coeficiente de determinación (�!).
- Interpretar los coeficientes obtenidos y su significado en el contexto del problema.
- Evaluar los resultados del modelo desde una perspec@va probabilís@ca

4. Clasificación [15 puntos]:
- E@quetar los ra@ngs como posi@vos (por encima de 3.5), neutrales (entre 2.5 y 3.5), y nega@vos
(por debajo de 2.5), asegurando que las clases estén balanceadas o u@lizando técnicas de
resampling.
- Entrenar los mismos modelos de clasificación u@lizados en la sección binaria (Regresión Logís@ca,
kNN, SVM, Naive Bayes, Árboles de Decisión, Random Forests), ajustados para clasificación
mul@clase.
- U@lizar validación cruzada para op@mizar hiperparámetros.
- Reportar las matrices de confusión para evaluar el rendimiento en cada clase. Evaluar las métricas
de accuracy, precision, recall y F1-Score.
- Comparar los resultados de los modelos binarios con los modelos mul@clase.

5. Op@mización de hiperparámetros [15 puntos]:
- Aplicar técnicas de op@mización de hiperparámetros (e.g., Grid Search, Random Search) para
encontrar los valores óp@mos de los hiperparámetros de los modelos LASSO y Ridge.
- Evaluar el rendimiento de los modelos con los hiperparámetros op@mizados y comparar con los
resultados anteriores.

6. Conclusiones [10 puntos]:
- Presentar conclusiones relevantes sobre los resultados y el trabajo realizado.
- Discu@r las ventajas y desventajas de cada modelo u@lizado, y su aplicabilidad en el contexto de
predicción de ra@ngs de bebidas alcohólicas.
- Proponer posibles mejoras o extensiones al análisis realizado

7. Crédito Extra: Regresión kNN [10 puntos]
- Entrenar un modelo de regresión kNN, incluyendo el proceso de op@mización del hiperparámetro
k u@lizando 5-fold cross valida@on.
- Comparar los resultados del modelo kNN con los resultados anteriores.

8. Crédito Extra: Regresión No Lineal [10 puntos]
- Evaluar modelos de Regresión No Lineal (polinómica u otras variantes), incluyendo el proceso de
op@mización de hiperparámetros u@lizando 5-fold cross valida@on.
- Comparar los resultados del no lineal con los resultados anteriores.